# D09 주식 발행내역 API 검증

## 목적
금융위원회 주식발행정보 API를 이용하여
특정 기업의 주식 발행내역을 조회할 수 있는지 검증한다.

### 검증 항목
1. 기업별 주식 발행내역 조회 가능 여부
2. 주식 발행일 확인
3. 발행차수 확인
4. 주식 발행사유 확인
5. 발행 이벤트를 DART 공시와 대조할 수 있는지 검토

### 참고
- 제공기관: 금융위원회
- 원천기관: 한국예탁결제원(KSD)
- API 기능: 주식발행내역조회
- Endpoint: getStocIssuInfo_V3
- 실시간 데이터가 아닌 일별 갱신 데이터

In [ ]:
import requests
import pandas as pd
from getpass import getpass

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", None)

In [ ]:
SERVICE_KEY = getpass("공공데이터포털 Decoding 인증키 입력: ")

BASE_URL = (
    "https://apis.data.go.kr/1160100/"
    "GetStocIssuInfoService_V3/getStocIssuInfo_V3"
)

COMPANY_NAME = "삼성전자"

In [ ]:
def get_stock_issue_info(
    service_key,
    company_name=None,
    crno=None,
    base_date=None,
    num_rows=100
):
    """
    금융위원회 주식발행내역 API 호출
    """

    params = {
        "serviceKey": service_key,
        "pageNo": 1,
        "numOfRows": num_rows,
        "resultType": "json",
    }

    # 너무 넓은 조회를 피하기 위해 가능한 조건을 넣음
    if company_name is not None:
        params["stckIssuCmpyNm"] = company_name

    if crno is not None:
        params["crno"] = crno

    if base_date is not None:
        params["basDt"] = base_date

    try:
        response = requests.get(
            BASE_URL,
            params=params,
            timeout=(5, 90)
        )

        print("HTTP 상태코드:", response.status_code)

        if response.status_code != 200:
            print("API 호출 실패")
            print(response.text[:2000])
            return pd.DataFrame()

        data = response.json()

        if "response" not in data:
            print("정상적인 API 응답 구조가 아닙니다.")
            print(data)
            return pd.DataFrame()

        header = data["response"].get("header", {})

        print(
            "API 결과:",
            header.get("resultCode"),
            header.get("resultMsg")
        )

        body = data["response"].get("body", {})
        items = body.get("items", {})

        if not items:
            print("조회 결과가 없습니다.")
            return pd.DataFrame()

        rows = items.get("item", [])

        if isinstance(rows, dict):
            rows = [rows]

        return pd.DataFrame(rows)

    except requests.exceptions.ReadTimeout:
        print("서버 응답시간 초과(ReadTimeout)")
        return pd.DataFrame()

    except requests.exceptions.RequestException as e:
        print("API 요청 오류:", e)
        return pd.DataFrame()

In [ ]:
df = get_stock_issue_info(
    service_key=SERVICE_KEY,
    company_name=COMPANY_NAME,
    num_rows=100
)

print("조회 건수:", len(df))

if not df.empty:
    display(df.head(20))

In [ ]:
print("===== D09 실제 반환 컬럼 =====")

for col in df.columns:
    print(col)

In [ ]:
date_like_cols = [
    col for col in df.columns
    if "Dt" in col or "date" in col.lower()
]

print("날짜 관련 후보 컬럼:")
print(date_like_cols)

In [ ]:
display(df.head(10))

In [ ]:
df["stckIssuDt"] = pd.to_datetime(
    df["stckIssuDt"],
    format="%Y%m%d",
    errors="coerce"
)

df = df.sort_values(
    "stckIssuDt",
    ascending=False
)

display(df.head(10))

In [ ]:
keywords = [
    "issu",     # 발행
    "rsn",      # reason
    "sq",       # 차수
    "cmpy",     # 회사
    "crno",     # 법인번호
    "basDt"     # 기준일
]

candidate_cols = [
    col for col in df.columns
    if any(k.lower() in col.lower() for k in keywords)
]

print("발행 관련 후보 컬럼:")
print(candidate_cols)

if candidate_cols:
    display(df[candidate_cols].head(10))

## 검증 포인트

D09의 최종 목적은 단순 API 호출 성공 여부가 아니라,
주식 발행 이벤트를 다른 데이터와 연결할 수 있는지 확인하는 것이다.

예상 활용 구조:

주식발행내역 API
→ 발행일
→ 발행사유
→ 발행차수
→ 동일 시점 DART 공시 검색
→ 발행 이벤트의 공시 근거 확인

In [ ]:
print("===== D09 기본 검증 =====")

print("API 데이터 존재:", not df.empty)
print("조회 건수:", len(df))

if not df.empty:
    print("전체 컬럼 수:", len(df.columns))
    print("중복 행 수:", df.duplicated().sum())

## D09 검증 결과

| 검증 항목 | 결과 |
|---|---|
| API 연결 | 확인 |
| 기업별 발행내역 조회 | 확인 |
| 발행일 정보 | 응답 필드 확인 후 기록 |
| 발행차수 | 응답 필드 확인 후 기록 |
| 발행사유 | 응답 필드 확인 후 기록 |
| DART 공시 대조 가능성 | 추후 검증 |

### 주의사항

1. 주식발행정보 API는 실시간 서비스가 아니다.
2. 발행내역과 DART 공시는 별도 데이터이므로 이벤트 날짜 및 기업 식별값을 기준으로 대조가 필요하다.
3. 발행내역이 존재하지 않는 것은 API 오류와 동일하지 않다.
4. API 응답의 실제 필드명은 공식 응답 결과를 기준으로 기록한다.
5. 본 데이터는 공공누리 제2유형으로 출처표시 및 상업적 이용 제한 조건이 적용된다.